In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2
import math
import random
import os
from collections import Counter

def Experiment(N, M, r):
    exp = 0
    t_N = N
    t_M = M
    for _ in range(r):
        tmp = random.random()
        if tmp <= t_M / t_N:
            exp += 1
            t_M -= 1
        t_N -= 1
    return exp

def play(N, M, r, num_simulations):
    return [Experiment(N, M, r) for _ in range(num_simulations)]

def theoretical_distribution(N, M, r):
    distribution = {}
    for k in range(min(M, r) + 1):
        c1 = math.comb(M, k)
        c2 = math.comb(N - M, r - k)
        c3 = math.comb(N, r)
        distribution[k] = c1 * c2 / c3
    return distribution

def empirical_distribution(results):
    count = Counter(results)
    total = len(results)
    empirical_dist = {k: v / total for k, v in count.items()}

    # Проверка суммы эмпирических вероятностей
    if abs(sum(empirical_dist.values()) - 1) > 1e-6:
        raise ValueError("Ошибка: сумма эмпирических вероятностей не равна 1.")

    return empirical_dist

def main():
    os.system('cls' if os.name == 'nt' else 'clear')
    print("####################")
    print("Содержание задачи:\nВ лотерее среди N билетов M выигрышных. Игрок покупает r билетов. С.в. η — число выигрышных билетов среди купленных\n")
    print("####################\n\n")

    N = int(input("Введите количество билетов: "))
    if N < 0:
        raise Exception("Неверный аргумент")
    M = int(input("Введите количество выигрышных билетов: "))
    if M < 0 or N < M:
        raise Exception("Неверный аргумент")
    r = int(input("Введите количество вытягиваемых билетов: "))
    if r < 0 or r > N:
        raise Exception("Неверный аргумент")
    num_simulations = int(input("Введите количество симуляций: "))
    if num_simulations < 0:
        raise Exception("Неверный аргумент")
    alpha = float(input("Введите уровень значимости alpha: "))
    k = int(input("Введите количество интервалов k: "))
    if k <= 0:
        raise Exception("Неверный аргумент")

    accept = 0
    reject = 0

    for t in range(1000):
        results = play(N, M, r, num_simulations)
        df = pd.DataFrame({'Вытянутые удачные билеты': results})

        theoretical_dist = theoretical_distribution(N, M, r)
        empirical_dist = empirical_distribution(results)

        yi = list(theoretical_dist.keys())
        ni = [results.count(y) for y in yi]

        # Проверка суммы ni
        if sum(ni) != num_simulations:
            print("Error")
            return

        theory_prob = [theoretical_dist[y] for y in yi]

        R0 = 0
        sumqk = 0
        tmp_prev = 0
        tmp = 0
        q = []


        min_last_probability = 0.01 / k

        for i in range(k):
            qk = 0
            nk = 0
            if i == k - 1:
                qk = max(1 - sumqk, min_last_probability)
                tmp = len(yi)
            else:
                while qk < 1 / k and tmp < len(theory_prob):
                    qk += theory_prob[tmp]
                    tmp += 1
            for j in range(tmp_prev, tmp):
                nk += ni[j]
            tmp_prev = tmp
            sumqk += qk
            q.append(qk)
            if num_simulations * qk != 0:
                R0 += (nk - num_simulations * qk) ** 2 / (num_simulations * qk)

        chi_square_value = chi2.ppf(1 - alpha, k - 1)
        if R0 <= chi_square_value:
            accept += 1
        else:
            reject += 1

    print('Гипотезу Ho приняли', accept, 'раз')
    print('Гипотезу Ho отклонили', reject, 'раз')

if __name__ == "__main__":
    main()

####################
Содержание задачи:
В лотерее среди N билетов M выигрышных. Игрок покупает r билетов. С.в. η — число выигрышных билетов среди купленных

####################


Введите количество билетов: 5
Введите количество выигрышных билетов: 2
Введите количество вытягиваемых билетов: 4
Введите количество симуляций: 10
Введите уровень значимости alpha: 1
